# MiniMax H3 (Hailuo 3.0) — Architecture, capacités… et une bifurcation juridique

> **Verdict SOTA : INTRINSIC pour l'auto-hébergement UE + voie service cloud ouverte.** Ce notebook sépare deux instruments juridiques distincts qui gouvernent MiniMax H3 :
>
> - **La *MiniMax H3 Community License*** (les **poids** téléchargeables) interdit explicitement l'usage, l'hébergement **et l'affichage des Outputs** dans l'Union Européenne (territoire exclu) → **INTRINSIC, définitif** pour tout déploiement auto-hébergé en UE (Sections 1 à 5).
> - **Les *Terms of Service* de la plateforme MiniMax / Hailuo** (le **service** hébergé, souscrit et facturé) sont un **second instrument** : un contrat de service souscrit depuis la France n'hérite pas d'une licence de poids qu'on n'a jamais acceptée. Ces ToS ne contiennent **aucune exclusion UE** (Section 6, lue *firsthand* ; l invocation concrète se fait dans [`04-5-MiniMax-H3-Cloud-Video`](../04-Applications/04-5-MiniMax-H3-Cloud-Video.ipynb)).
>
> Ce couple de verdicts est *plus* honnête qu'un INTRINSIC global : il dit exactement ce qui est fermé (auto-hébergement) et ce qui reste ouvert (invocation du service souscrit). Cette transparence est elle-même une compétence d'ingénierie ML.

**Sources officielles** :
- `MiniMax H3 COMMUNITY LICENSE AGREEMENT` (date de licence : 2 août 2026), déposée sur [HuggingFace `MiniMaxAI/MiniMax-H3`](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE).
- Code modèle : [GitHub `MiniMax-AI/MiniMax-H3`](https://github.com/MiniMax-AI/MiniMax-H3).
- *Terms of Service* du service (Section 6) : [minimax.io/terms-of-service-v2](https://www.minimax.io/terms-of-service-v2.html), [Hailuo Video ToS](https://hailuoai.video/doc/terms-of-service.html), [MiniMax Open Platform ToS](https://platform.minimax.io/protocol/terms-of-service) — lues *firsthand*, articles cités *verbatim*.


## Le contexte : un « Sora at home » qui fait vibrer la communauté

MiniMax H3 (aussi *Hailuo 3.0* / 海螺3), open-sourcé fin juillet / début août 2026, est arrivé **#1 du benchmark *Artificial Analysis* video editing** (Elo ~1130) au lancement. Sa promesse séduit : un modèle **omni-modal unifié** (texte, image, vidéo, audio en entrée) produisant de la vidéo jusqu'à **2K / 15 s / 24 fps** avec **audio stéréo natif** (32 kHz, 11 langues) — le tout en *open-weights*, donc en principe auto-hébergeable. C'est le « Sora at home » : la même ambition qu'OpenAI Sora, mais en poids téléchargeables plutôt qu'en API fermée.

Ce notebook examine pourquoi, **pour notre contexte (France / UE, dépôt public, écoles partenaires)**, cette promesse se heurte à un obstacle qui n'est pas technique mais **juridique** — et comment raisonner sobrement cette décision de déploiement.


## Section 1 — La licence *MiniMax H3 Community License* : une restriction territoriale explicite

Lisons les clauses pivots (verbatim, numérotation originale) :

> **Art. I.5 — « Excluded Territories »** : *means the European Union, the United Kingdom, the Republic of Korea and the United States of America.*
>
> **Art. I.3 — « Applicable Territory »** : *means worldwide, excluding the Excluded Territories.*
>
> **Art. II — Grant of Rights** : *Solely within the Applicable Territory, we grant you a non-exclusive, non-transferable, royalty-free, limited license to use, reproduce, distribute, create derivative works […] and modify the Materials […].* **Aucune exception pour l'éducation, la recherche ou un usage non-commercial** n'est prévue.
>
> **Art. V.4 — Use Restrictions** : *You may not use, reproduce, modify, distribute, or display the MiniMax H3 Works **or any of their Outputs or results** outside the Applicable Territory. Any such use outside the Applicable Territory is not authorized by this Agreement.*
>
> **Exhibit A.1** (Acceptable Use Policy, première utilisation interdite) : *Use outside the Applicable Territory.*

Deux points sont souvent mal compris et méritent d'être soulignés :

1. **Les *Outputs* sont couverts, pas seulement les poids** — *pour quiconque exécute les poids*. La licence couvre les *Works* « y compris via tout *Hosted Service* » (Art. I.8). Donc si **vous** téléchargez et exécutez les poids H3 (y compris via un service que **vous** hébergez), afficher ou redistribuer la vidéo générée dans l'UE est un usage *outside the Applicable Territory* → non autorisé. L'argument « on n'héberge pas les poids chez nous, on appelle juste l'API » **ne tient pas** *tant qu'on reste dans le périmètre de cette licence des poids*.
   - ⚠️ **Mais attention** : un *consommateur* du service cloud Hailuo exploité par MiniMax n'**accepte jamais** la Community License (il ne télécharge pas les poids). Il est lié par un **autre instrument** — les *Terms of Service* de la plateforme — qui, lui, ne contient **aucune exclusion UE**. Cette **bifurcation** (licence des poids ≠ ToS du service) est le sujet de la **Section 6**. Ne la skippez pas.
2. **L'« open-weights » n'est pas le « open source ».** Les poids sont téléchargeables, mais sous une licence propriétaire à restriction territoriale — l'opposé d'une licence OSI. Confondre les deux est une erreur classique de mise en production.

La seule voie vers un usage UE des **poids** est une **licence commerciale séparée** (Art. II, dernier § : *« you are welcome to contact us about obtaining a license »*) — une démarche *provider-side* qui dépend de MiniMax (Nanonoble Pte. Ltd.), pas de l'utilisateur.


## Section 2 — Vérificateur de juridiction : coder la conformité

Plutôt qu'un discours, codons le raisonnement. La fonction suivante **parse la clause *Excluded Territories*** et décide, pour une localisation de déploiement donnée, si l'usage de MiniMax H3 y est autorisé. Ce code tourne localement (aucun appel au modèle) — il analyse le texte de licence, pas le modèle.


In [1]:
# Excluded Territories au sens de l'Art. I.5 (verbatim license).
# Couverture par code pays / région ISO, pour le raisonnement de conformité.
EXCLUDED_TERRITORIES = {
    "European Union": {
        "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR",
        "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK",
        "SI", "ES", "SE",
    },
    "United Kingdom": {"GB"},
    "Republic of Korea": {"KR"},
    "United States of America": {"US"},
}


def is_applicable_territory(country_code: str) -> bool:
    """True si country_code (ISO-3166 alpha-2) est dans l'Applicable Territory.

    L'Applicable Territory = monde entier hors Excluded Territories (Art. I.3).
    """
    cc = country_code.upper()
    for states in EXCLUDED_TERRITORIES.values():
        if cc in states:
            return False
    return True


def deployment_verdict(country_code: str, label: str = "") -> str:
    """Rend un verdict lisible pour un déploiement envisagé dans country_code."""
    ok = is_applicable_territory(country_code)
    where = label or country_code
    if ok:
        return f"{where} ({country_code}) : Applicable Territory — usage autorise sous la Community License"
    # identifier quel territoire exclu
    for name, states in EXCLUDED_TERRITORIES.items():
        if country_code.upper() in states:
            return f"{where} ({country_code}) : EXCLUDED ({name}) — usage NON autorise, licence commerciale requise"
    return f"{where} ({country_code}) : indetermine"


# --- Verdicts pour nos contextes reels ---
contextes = [
    ("FR", "France (user + ecoles EPITA/ECE/ESGF/EPF)"),
    ("DE", "Allemagne"),
    ("GB", "Royaume-Uni"),
    ("US", "Etats-Unis"),
    ("KR", "Coree du Sud"),
    ("CN", "Chine"),
    ("JP", "Japon"),
    ("BR", "Bresil"),
]
for cc, label in contextes:
    print(deployment_verdict(cc, label))


France (user + ecoles EPITA/ECE/ESGF/EPF) (FR) : EXCLUDED (European Union) — usage NON autorise, licence commerciale requise
Allemagne (DE) : EXCLUDED (European Union) — usage NON autorise, licence commerciale requise
Royaume-Uni (GB) : EXCLUDED (United Kingdom) — usage NON autorise, licence commerciale requise
Etats-Unis (US) : EXCLUDED (United States of America) — usage NON autorise, licence commerciale requise
Coree du Sud (KR) : EXCLUDED (Republic of Korea) — usage NON autorise, licence commerciale requise
Chine (CN) : Applicable Territory — usage autorise sous la Community License
Japon (JP) : Applicable Territory — usage autorise sous la Community License
Bresil (BR) : Applicable Territory — usage autorise sous la Community License


## Section 3 — Architecture : ce qui fait la force du modèle (et que nous ne pouvons pas exercer ici)

Bien que nous ne puissions pas exécuter H3, comprendre son architecture est légitime et précieux. Le dépôt [`MiniMax-AI/MiniMax-H3`](https://github.com/MiniMax-AI/MiniMax-H3) révèle les composants clés :

| Composant (dépôt) | Rôle | Détail technique |
|---|---|---|
| **`FL2VA`** | *Flow-Language-to-Video-Audio* — le cœur omni-modal | Encode unifie texte/image/vidéo/audio en entrée vers une représentation latente partagée |
| **`Ref2VA`** | *Reference-to-Video-Audio* | Gère les références multiples (first+last frame, omni-reference) pour l'édition instructionnelle |
| **`transformer`** / **`transformer_ref`** | Backbone de débruitage diffusif | Le transformeur principal + la branche de conditionnement par référence |
| **`vae`** | *Variational Autoencoder* vidéo | Compresse les pixels vidéo vers l'espace latent (jusqu'à 2K) |
| **`audio_vae`** + **`audio_scheduler`** | **Audio stéréo natif** | La différence distinctive : génère l'audio synchronisé (32 kHz, 11 langues) *en même temps* que la vidéo, pas en pass séparé |
| **`text_encoder`** + **`tokenizer`** | Encodage du prompt | Le `processor` orchestre l'entrée omni-modale |
| **`scheduler`** | Planificateur de diffusion | Contrôle le nombre de pas de débruitage |

**Point de licence notable** (Additional Note du LICENSE) : l'encodeur de H3 utilise **Qwen3-VL-32B**, lui-même sous **Apache 2.0** (une licence réellement ouverte, sans restriction territoriale). Mais cet encodeur est un *constituant* de H3 — l'assemblage complet reste sous la Community License géo-restreinte. On ne peut pas « récupérer juste l'encodeur Apache » pour contourner la restriction : le modèle intégré est un *Material* couvert par l'Accord.

**Pourquoi c'est un excellent cas pédagogique** : H3 combine une capacité absente de nos autres notebooks (vidéo 2K **+ audio natif synchronisé** dans un seul modèle omni-modal) — exactement le genre de prouesse technique qu'un cours SOTA voudrait montrer. La licence nous oblige à le faire *différemment* : par l'analyse, pas l'exécution.


## Section 4 — Matrice de décision : H3 vs ses alternatives

Face à une licence géo-restreinte, la compétence clé est de **raisonner en alternatives**. Le code suivant construit une matrice comparant MiniMax H3 aux autres modèles vidéo de la série (et à Sora), pour aider à décider quoi utiliser selon la juridiction.


In [2]:
# Matrice de decision : modeles video de la serie + Sora, selon 5 axes de conformite/usage.
# 'geo_restricted' = True si la licence exclut des territoires (UE en particulier).
MODELES = [
    {
        "nom": "MiniMax H3",
        "open_weights": True,
        "geo_restricted_ue": True,
        "audio_natif": True,
        "resolution_max": "2K",
        "notebook_series": "02-6 (ce notebook, descriptif)",
        "licence": "MiniMax H3 Community License (UE exclue)",
    },
    {
        "nom": "Wan 2.1/2.2",
        "open_weights": True,
        "geo_restricted_ue": False,
        "audio_natif": False,
        "resolution_max": "1080p",
        "notebook_series": "02-3 (executable localement)",
        "licence": "Apache 2.0 (permissive)",
    },
    {
        "nom": "HunyuanVideo",
        "open_weights": True,
        "geo_restricted_ue": False,
        "audio_natif": False,
        "resolution_max": "1080p",
        "notebook_series": "02-1 (executable localement)",
        "licence": "Tencent Community License (permissive UE)",
    },
    {
        "nom": "LTX-2",
        "open_weights": True,
        "geo_restricted_ue": False,
        "audio_natif": True,
        "resolution_max": "1080p",
        "notebook_series": "02-5 (executable, audiovisuel conjoint)",
        "licence": "LTX-2 Community License (permissive UE)",
    },
    {
        "nom": "OpenAI Sora",
        "open_weights": False,
        "geo_restricted_ue": False,
        "audio_natif": True,
        "resolution_max": "1080p",
        "notebook_series": "04-3 (API cloud, sous quota/cout)",
        "licence": "API fermee (ToS OpenAI)",
    },
]


def utilisables_en_ue(modeles):
    """Filtre les models utilisables en UE (non geo-restreints UE)."""
    return [m for m in modeles if not m["geo_restricted_ue"]]


def avec_audio_natif(modeles):
    """Modeles offrant l'audio natif synchronise."""
    return [m["nom"] for m in modeles if m["audio_natif"]]


print("=== Modeles utilisables en UE (pas de restriction territoriale UE) ===")
for m in utilisables_en_ue(MODELES):
    print(f"  - {m['nom']:<16} | {m['licence']}")

print()
print("=== Modeles avec audio natif synchronise ===")
print("  " + ", ".join(avec_audio_natif(MODELES)))

print()
print("=== Alternatives UE a H3 pour le cas 'video + audio natif' ===")
# H3 est exclus: on cherche les modeles utilisables en UE AVEC audio natif.
alt = [m for m in utilisables_en_ue(MODELES) if m["audio_natif"]]
for m in alt:
    print(f"  - {m['nom']:<16} | res {m['resolution_max']} | notebook {m['notebook_series']}")


=== Modeles utilisables en UE (pas de restriction territoriale UE) ===
  - Wan 2.1/2.2      | Apache 2.0 (permissive)
  - HunyuanVideo     | Tencent Community License (permissive UE)
  - LTX-2            | LTX-2 Community License (permissive UE)
  - OpenAI Sora      | API fermee (ToS OpenAI)

=== Modeles avec audio natif synchronise ===
  MiniMax H3, LTX-2, OpenAI Sora

=== Alternatives UE a H3 pour le cas 'video + audio natif' ===
  - LTX-2            | res 1080p | notebook 02-5 (executable, audiovisuel conjoint)
  - OpenAI Sora      | res 1080p | notebook 04-3 (API cloud, sous quota/cout)


## Section 5 — Cadre de décision : « puis-je légalement déployer ce modèle ici ? »

MiniMax H3 illustre un cadre de raisonnement général, utile pour **tout** modèle qu'on envisage d'intégrer en production ou en enseignement :

1. **Lire la licence à la source** (pas un résumé secondaire). Ici, le `LICENSE` brut HuggingFace — pas un article de blog.
2. **Identifier les restrictions structurelles** : territoriales (H3), de revenu commercial (H3 Art. IV : seuil 20 M$), d'usage (Exhibit A : pas de military, pas de désinformation, etc.).
3. **Vérifier la portée** : la restriction couvre-t-elle seulement les poids, ou aussi les *Outputs* et les *Hosted Services* ? (H3 : les trois.)
4. **Croiser avec son contexte** : juridiction (France = UE), public (ici : dépôt public + écoles = large diffusion), usage (commercial vs éducatif — mais H3 n'exempte pas l'éducatif).
5. **Décider entre** : (a) obtenir une licence commerciale, (b) usage descriptif/analytique sans exécuter ni afficher d'Outputs, (c) choisir une alternative permissive.

Ce notebook incarne l'option **(b)** appliquée à l'enseignement : étudier le modèle sans l'exécuter. C'est honnête (verdict INTRINSIC documenté, pas de contournement) et pédagogiquement riche (la décision de conformité *est* la leçon).


## Section 6 — La bifurcation juridique : licence des poids ≠ ToS du service

La Section 5 conclut l'analyse de la *Community License* (les poids). Mais H3 est aussi accessible via un **service cloud hébergé** par MiniMax (la plateforme *Hailuo* / *Hailuo AI*), que l'on peut *consommer* sans jamais télécharger les poids. **Ces deux voies sont régies par deux instruments juridiques distincts** — c'est l'erreur de raisonnement la plus coûteuse en production que de les confondre.

| Instrument | Ce qu'il régit | Lu *firsthand* ? | Exclusion UE ? |
|---|---|---|---|
| **MiniMax H3 Community License** (HF `MiniMaxAI/MiniMax-H3`) | les **poids** : usage, hébergement, exécution, Outputs de *votre* exécution | **OUI** (Section 1) | **OUI** — UE exclue (Art. I.5, définitif) |
| **minimax.io Terms of Service v2** (parapluie international) | tous les services MiniMax | **OUI** (cette section) | **NON** — seule mention : *« geographic availability »* souple |
| **Hailuo Video Terms of Service** (`hailuoai.video`, le service vidéo souscrit) | la plateforme de génération vidéo Hailuo | **OUI** (cette section) | **NON** — opérateur Singapour ; accès UE à vos risques (lois locales) |
| **MiniMax Open Platform Terms of Service** (`platform.minimax.io`, la voie API) | l'accès développeur / API programmatique | **OUI** (cette section) | **NON** — Nanonoble Pte. Ltd. (Singapour) ; France ≠ *Restricted Person* |

### Le service ne contient **aucune** exclusion UE — lues *firsthand*

Clés pivot, citées *verbatim* (standard = même rigueur que pour la licence des poids) :

**Hailuo Video ToS — Geographic Restrictions** :
> *"The Services is owned and operated by a Singapore company. […] Access to the Services may be unlawful for certain individuals or in specific countries. **If you choose to access the Services from outside Singapore, you do so at your own initiative and are solely responsible for ensuring compliance with applicable local laws.**"*

Aucune exclusion nominative de l'UE, de la France, du Royaume-Uni ou des États-Unis. La clause est une **responsabilité** (respectez vos lois locales — ici : la loi française, qui n'interdit pas l'usage d'un service de génération vidéo souscrit), pas une **interdiction**.

**Hailuo Video ToS — User Generated Content (Outputs)** :
> *"We do not claim ownership of User Contributions or User Generated Content."*

L'utilisateur **possède** ses Outputs — mais la phrase qui suit immédiatement dans la même section accorde à MiniMax une **licence** sur ce contenu :

> *"By using the Services, you grant to us [...] a **royalty-free, perpetual, irrevocable, worldwide, non-exclusive** right [...] to use, license, reproduce, modify, adapt, **publish**, translate, create derivative works from, **distribute** [...] your User Contributions and User Generated Content"*, et *"This license survives termination of this Agreement by any party, for any reason."*

Cette cession ne **retire pas** à l'utilisateur le droit de diffuser ses propres Outputs (la voie UE reste ouverte), mais elle **coexiste** avec sa propriété : un notebook dont le *sujet* est d'apprendre à lire une licence doit citer la clause **entière** — la moitié flatteuse (*« we do not claim ownership »*) et la cession qui la suit (*« royalty-free, perpetual, irrevocable... publish, distribute »*) vivent ensemble sans se contredire, et c'est précisément ce qu'un étudiant doit apprendre à voir. **Aucune restriction territoriale sur l'affichage ou la distribution des Outputs** — c'est la différence radicale avec l'Art. V.4 de la licence des poids.

**MiniMax Open Platform ToS — Outputs (voie API)** :
> *"As between you and us, and to the extent permitted by applicable laws, **you retain your ownership rights in Client input and generated content.**"*

Même conclusion sur la voie API : l'utilisateur garde ses droits sur les Outputs, sans restriction territoriale. La France n'est ni un *Restricted Person* (listes de sanctions OFAC/ONU/UK/UE — la clause liste ces listes comme *obligation de conformité pour l'utilisateur*, pas comme exclusion de l'utilisateur) ni un *Sanctioned Region* (Iran, Cuba, DPRK, Syrie, Crimée/Donetsk/Luhansk).

### Pourquoi le non-héritage est juridiquement cohérent

La Community License lie **celui qui l'accepte** — c'est-à-dire qui télécharge et utilise les *Materials* (poids). Le contrat de service lie **celui qui s'abonne** — un consommateur qui n'a jamais accepté la licence des poids. MiniMax (Nanonoble Pte. Ltd.) détient les droits sur les poids H3 (elle en est le créateur) et **exploite** le service Hailuo *sous ses propres ToS*, en assumant elle-même ses obligations de licence. Le consommateur UE contracte avec MiniMax via les ToS du service, pas via la Community License.

### Résidus honnêtes (à ne pas survendre)

Cette conclusion est **supportée** par la lecture *firsthand* des trois instruments de service + par l'existence d'une souscription UE active. Trois résidus la qualifient sans l'invalider :

1. **Une AUP (*Acceptable Use Policy*) autonome** est référencée partout comme gouvernant l'usage des Outputs, mais l'auteur n'a pas localisé d'URL officielle *firsthand* (les agrégateurs tiers qui la reproduisent ne font pas foi). L'Open Platform ToS contient des clauses *inline* de conduite prohibée (sécurité nationale, fraude, *infringement* IP, sanctions) qui en tiennent lieu pour le service. D'après les résumés secondaires (non *firsthand*), cette AUP est du **conduite** (usages militaires, impersonation, contenu AI non divulgué en public) — **pas territorial**. À confirmer sur la source officielle.
2. **Une clause région-États-Unis spécifique** existe dans l'Open Platform ToS : *"If you select the United States as the service region, you […] may not: […]"*. Elle est **déclenchée par la sélection de région = US** — inapplicable à un utilisateur France opérant en région non-US. C'est un garde-fou anti-contournement pour le marché US, pas un ban UE.
3. **Un sign-off légal formel** (validation conforme par un conseil juridique, ou confirmation écrite de MiniMax) irait au-delà de la lecture publique des ToS. Ce notebook documente ce que les ToS **publiques** disent — suffisant pour une décision de conception pédagogique, pas un avis juridique.

**Verdict net** : la voie service cloud Hailuo est **ouverte depuis la France** sous les ToS du service (aucune exclusion UE, Outputs sans restriction territoriale). C'est un **second instrument**, distinct de la Community License qui reste INTRINSIC pour l'auto-hébergement. Le notebook **[`04-5-MiniMax-H3-Cloud-Video`](../04-Applications/04-5-MiniMax-H3-Cloud-Video.ipynb)** (famille *Applications / cloud-API*) montre comment l'invoquer concrètement : loader idempotent (sans rebrûler de quota), scénarios omni-modaux en HD/2K avec audio natif — ce que l'auto-hébergement UE ne peut pas faire, et que le service seul achète *sans rebrûler de quota* et *sans automatisation de compte consommateur*.


## Exercices

Les trois exercices suivants s'exercent sur la **décision de conformité** — la compétence distinctive de ce notebook. Aucun n'exécute le modèle (la licence l'interdit). Complétez les stubs.


### Exercice 1 — Étendre le vérificateur aux *Hosted Services* (API)

La licence couvre aussi l'usage via API (Art. I.8 *Hosted Services*). Écrivez `can_call_api(country_code, provider_region)` qui renvoie `False` si **soit** le pays de l'appelant, **soit** la région du provider est un territoire exclu — car l'Output serait consommé/affiché dans un territoire exclu.


In [3]:
def can_call_api(country_code: str, provider_region: str) -> bool:
    """True si un appel API MiniMax H3 est autorise pour cet appelant + ce provider.

    Rappel (Art. V.4) : les Outputs ne peuvent etre ni utilises ni affiches hors
    Applicable Territory. Donc l'appel est bloque des que l'appelant OU le provider
    est en territoire exclu.
    """
    # Indice : reutilisez is_applicable_territory(country_code) definie plus haut.
    # Etape 1 : verifier country_code (ou l'appelant consomme l'Output).
    # Etape 2 : verifier provider_region (ou l'Output est produit/affiche).
    # Etape 3 : retourner True seulement si les DEUX sont dans l'Applicable Territory.
    return None  # TODO etudiant


# Test rapide (a decommenter) :
# print(can_call_api("FR", "CN"))   # attendu : False (FR = UE exclue)
# print(can_call_api("JP", "CN"))   # attendu : True (les deux applicables)
# print(can_call_api("JP", "US"))   # attendu : False (US exclue)


### Exercice 2 — Sélectionneur de modèle conforme

Écrivez `choisir_modele_conforme(exigences)` qui, étant donné un dictionnaire d'exigences (pays, `audio_natif` booléen, `open_weights` booléen), renvoie la **liste** des modèles de `MODELES` qui satisfont **toutes** les contraintes — typiquement pour recommander une alternative UE à H3.


In [4]:
def choisir_modele_conforme(exigences: dict) -> list:
    """Renvoie les modeles de MODELES satisfaisant toutes les exigences.

    exigences cles possibles :
      - 'pays' (ISO alpha-2) : le modele doit etre utilisable dans ce pays
      - 'audio_natif' (bool) : le modele doit (ou non) offrir l'audio natif
      - 'open_weights' (bool) : le modele doit (ou non) etre open-weights
    """
    # Indice : filtrez MODELES en cumulatif. Pour 'pays', ecartez les
    # geo-restreints UE si le pays est dans l'UE (is_applicable_territory == False).
    resultats = []
    # TODO etudiant
    return resultats


# Test rapide (a decommenter) :
# besoin = {"pays": "FR", "audio_natif": True, "open_weights": True}
# for m in choisir_modele_conforme(besoin):
#     print(f"  conforme : {m['nom']}")
# # attendu : LTX-2 (UE-ok, audio natif, open-weights). H3 exclu (UE), Sora exclu (pas open-weights).


### Exercice 3 — Détecteur de clause de territorialité dans une licence arbitraire

Écrivez `detecte_restriction_territoriale(texte_licence)` qui prend le texte brut d'une licence et renvoie la liste des noms de pays/régions mentionnés dans une clause d'exclusion géographique (regardez les motifs comme *« excluding »*, *« Excluded Territories »*, *« not permitted in »*). Ceci généralise le raisonnement au-delà de H3.


In [5]:
def detecte_restriction_territoriale(texte_licence: str) -> list:
    """Renvoie les entites geographiques citees dans une clause d'exclusion.

    Recherche des amorces de clause ('excluding', 'Excluded Territories',
    'not permitted in', 'not authorized in') puis extrait les noms de pays/regions
    connus qui suivent.
    """
    entites_connues = [
        "European Union", "United Kingdom", "United States", "United States of America",
        "Republic of Korea", "China", "Japan", "France", "Germany", "Brazil",
    ]
    trouvees = []
    # Indice : pour chaque entite connue, verifier si elle apparait dans le texte
    # A PROXIMITE d'une amorce de clause d'exclusion (pas n'importe ou : une licence
    # peut citer un pays pour d'autres raisons). Une approche simple : chercher
    # l'amorce, puis scanner les N caracteres qui suivent.
    # TODO etudiant
    return trouvees


# Test rapide (a decommenter) avec un extrait de la licence H3 :
# extrait = '''"Excluded Territories" means the European Union, the United Kingdom,
# the Republic of Korea and the United States of America.'''
# print(detecte_restriction_territoriale(extrait))
# # attendu : les 4 territoires exclus


## Conclusion — transparence et conformité comme compétences (verdict double)

MiniMax H3 est techniquement remarquable (#1 video editing, omni-modal, audio natif). Sa gouvernance se scinde en **deux instruments juridiques**, et ce notebook rend un verdict **double** :

| Voie | Instrument | Verdict | État |
|---|---|---|---|
| **Auto-hébergement** (poids) | *MiniMax H3 Community License* | **INTRINSIC** — UE exclue (Art. I.5), Outputs couverts (Art. V.4) | Définitif. Licence commerciale requise pour débloquer. |
| **Service cloud** (Hailuo souscrit) | *Hailuo Video ToS* + *Open Platform ToS* | **Ouverte** — aucune exclusion UE, Outputs sans restriction territoriale. Entitlement **par série** (probe 2026-08-10) : H3 plan-gated (400 TokenPlan 2013), `video-01` couvert | Notebooks dédiés : [`04-5`](../04-Applications/04-5-MiniMax-H3-Cloud-Video.ipynb) (H3, squelette idempotent) et [`04-6`](../04-Applications/04-6-MiniMax-video-01-v1-Cloud-Video.ipynb) (`video-01`/`v1`, voie de génération réelle). |

Ce couple est *plus* honnête qu'un INTRINSIC global : il nomme exactement ce qui est fermé (l'auto-hébergement) et ce qui reste ouvert (le service souscrit). Cinq leçons à retenir :

1. **Open-weights ≠ open-source ≠ libre d'usage partout.** Toujours lire la licence à la source et identifier les restrictions structurelles (territoriales, commerciales, d'usage).
2. **La portée compte autant que l'octroi.** Une licence peut autoriser un usage mais en restreindre l'affichage, la redistribution, ou les Outputs — comme la Community License des poids.
3. **Un modèle peut avoir plusieurs instruments juridiques.** La licence des *poids* et le contrat de *service* sont distincts ; classer l'un par héritage supposé de l'autre, sans lire le second, est l'erreur la plus coûteuse. C'est exactement ce que les étudiants rencontreront en entreprise.
4. **La conformité est une décision de conception, pas une après-pensée.** Choisir tôt entre (a) licence commerciale, (b) usage descriptif, (c) alternative permissive, (d) service souscrit — c'est ce que fait ce notebook.
5. **L'idempotence est une exigence de service, pas une politesse.** Un service sous quota (5 générations/jour) impose un loader idempotent (implanté dans le notebook [`04-5`](../04-Applications/04-5-MiniMax-H3-Cloud-Video.ipynb)) : artefact présent → chargé, génération → flag explicite. Une ré-exécution ne rebrûle jamais le quota.

**Trois régimes, trois verdicts — le triptyque de la série** : le même besoin (« générer une vidéo avec un modèle SOTA ») se présente sous trois régimes juridiques et techniques distincts dans la série, et chacun porte un verdict SOTA différent :

| Notebook | Modèle | Régime juridique | Exécution | Sortie |
|---|---|---|---|---|
| **`02-7`** (CogVideoX-2b) | Apache-2.0 | **locale**, RTX 3090, ~16 GB | 480×720 @ 8 fps, **muet** |
| **`02-5`** (LTX-2) | licence permissive, UE OK | **locale**, audiovisuel | vidéo + audio conjoints |
| **`02-6`** (MiniMax H3, ce notebook) | Community License, **UE exclue** | **impossible** (auto-hébergement) | — (descriptif, INTRINSIC) |
| **[`04-5`](../04-Applications/04-5-MiniMax-H3-Cloud-Video.ipynb)** (MiniMax H3, service) | *product terms* du service | **cloud**, 5 générations/jour — entitlement H3 plan-gated (TokenPlan 2013) | **HD/2K + audio stéréo natif** (à l'activation du plan) |
| **[`04-6`](../04-Applications/04-6-MiniMax-video-01-v1-Cloud-Video.ipynb)** (MiniMax video-01, service) | *product terms* du service | **cloud**, `/v1` — plan couvrant `video-01` (vérifié par probe 2026-08-10) | génération réelle (gate : provision de la clé à la lane) |
Le contraste `02-7` ↔ `04-5` **quantifie** exactement ce que le service cloud achète : 480×720@8fps muet en local permissif, contre HD/2K sonore par le service. C'est la charge pédagogique de H3 — le même modèle, deux voies, et le service seul débloque le format HD + l'audio natif omni-modal.

**États réversibles** :
- Si une licence commerciale UE est obtenue auprès de MiniMax → ce notebook évolue vers une exécution locale réelle (ComfyUI workflow + VRAM mesurée, comme `02-3-Wan`).
- En attendant, l'alternative UE pédagogiquement équivalente pour « vidéo + audio natif » est **LTX-2** (`02-5-LTX2-Audiovisual`, licence permissive, exécutable).
- La voie service cloud est juridiquement ouverte mais **gated en deux étapes** : provision de `MINIMAX_GENAI_API_KEY` (absente de la machine au 2026-08-15) **et** entitlement de la série (probe 2026-08-10 : H3 → 400 TokenPlan 2013 ; `video-01`/`v1` → couvert). Les loaders idempotents sont posés dans **[`04-5`](../04-Applications/04-5-MiniMax-H3-Cloud-Video.ipynb)** (H3) et **[`04-6`](../04-Applications/04-6-MiniMax-video-01-v1-Cloud-Video.ipynb)** (video-01).
- **Surveillance licence (re-vérifiée firsthand le 2026-08-15)** : le `LICENSE` de `MiniMaxAI/MiniMax-H3` (daté du 2 août 2026) exclut toujours l'UE — *« Excluded Territories » means the European Union, the United Kingdom, the Republic of Korea and the United States of America*. Aucun changement depuis la vérification initiale.

---
*Note : ce notebook n'embarque aucun poids H3 et n'affiche aucun Output de la *Community License* — conformément à l'Art. V.4. Les Outputs du service cloud (notebook [`04-5`](../04-Applications/04-5-MiniMax-H3-Cloud-Video.ipynb), lorsqu'activée) proviennent du service Hailuo souscrit, sous les *Terms of Service* de la plateforme (Section 6) — un instrument distinct, qui n'exclut pas l'UE. Le code des Sections 2-5 analyse le texte de licence et raisonne sur la conformité ; ses sorties sont produites localement par ce raisonnement.*
